# Análisis exploratorio — Colono IA

Recomendador de colocación inicial en Catan.

Este notebook explora el dataset de parejas de vértices generado por simulación y decide
qué variables entran al modelo. Corre de principio a fin sin errores.

**Datos:** `data/processed/parejas.csv`, generado con `python src/generar_dataset.py`.
Una fila por pareja de vértices candidata. La variable objetivo `puntos` es el promedio de
puntos de victoria de 40 mini-partidas simuladas empezando en esa pareja.

## 1. Preparación

In [ ]:
# En Google Colab, descomenta para clonar el repositorio:
# !git clone https://github.com/AnaaBohorquez/catanai.git
# %cd catanai

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

RUTA = "../data/processed/parejas.csv"   # en Colab: "data/processed/parejas.csv"
IDENTIFICADORES = ["tablero_id", "pareja_id"]
OBJETIVO = "puntos"

In [ ]:
datos = pd.read_csv(RUTA)
variables = [c for c in datos.columns if c not in IDENTIFICADORES + [OBJETIVO]]

print(f"Filas:     {len(datos):,}")
print(f"Tableros:  {datos['tablero_id'].nunique()}")
print(f"Parejas por tablero: {len(datos) / datos['tablero_id'].nunique():.0f}")
print(f"Variables: {len(variables)}")
print(f"Valores faltantes: {int(datos.isna().sum().sum())}")

datos.head(3)

### Por qué no hay valores faltantes

Los datos no vienen de una fuente externa: los genera el propio simulador. Cada variable se
calcula con una fórmula que siempre devuelve un número, y `variables.py` lo verifica con
`verificar_variables()`, que falla si aparece un `None`, un `NaN` o un infinito.

Esto ahorra el trabajo de imputación, pero traslada la exigencia a otro lado: hay que
comprobar que las variables **tengan sentido** y que **varíen**, que es lo que hace el
resto de este notebook.

## 2. La variable objetivo

In [ ]:
print(datos[OBJETIVO].describe().round(2).to_string())

fig = px.histogram(
    datos, x=OBJETIVO, nbins=60,
    title="Distribución de los puntos simulados",
    labels={OBJETIVO: "Puntos de victoria en 20 rondas"},
)
fig.update_layout(bargap=0.02, height=400)
fig.show()

La distribución tiene **sesgo a la derecha**: la mayoría de las parejas se agrupa entre 3 y 6
puntos, y una cola pequeña llega mucho más alto. Eso es lo esperable y es buena señal para el
producto: las parejas realmente buenas son raras, y encontrarlas es justo lo que la app aporta.

La escala llega hasta unos 13 puntos, coherente con una partida real de Catan, que se gana con 10.

## 3. Qué variables predicen los puntos

In [ ]:
correlaciones = (
    datos[variables + [OBJETIVO]]
    .corr(numeric_only=True)[OBJETIVO]
    .drop(OBJETIVO)
    .sort_values(key=abs, ascending=False)
)

fig = px.bar(
    correlaciones.head(20).iloc[::-1],
    orientation="h",
    title="Correlación con los puntos simulados (20 variables más fuertes)",
    labels={"value": "Correlación", "index": ""},
    color=correlaciones.head(20).iloc[::-1].values,
    color_continuous_scale="RdBu", range_color=[-0.7, 0.7],
)
fig.update_layout(height=620, showlegend=False, coloraxis_showscale=False)
fig.show()

correlaciones.round(3).to_frame("correlacion")

### Tres lecturas que importan

**`produccion_efectiva` gana a `pips_totales`.** Ajustar la producción por lo que realmente
puedes gastar, descontando el excedente por su tasa de cambio, predice bastante mejor que
contar pips a secas. Los pips son la medida que usa todo el mundo en la mesa, y resulta ser
de las flojas.

**El desequilibrio resta.** Su correlación es negativa: concentrar la producción en pocos
recursos hace daño, aunque el total de pips sea alto. Es el fenómeno de acumular cartas que
no se pueden gastar.

**Los puertos correlacionan en negativo, y eso es engañoso.** No es que los puertos sean malos:
están en la costa, y los vértices costeros tocan uno o dos hexágonos en vez de tres, así que
producen menos. La correlación simple mezcla las dos cosas. La regresión, al controlar por la
producción, debería separar el efecto propio del puerto. Es un buen ejemplo de por qué una
correlación no basta para concluir.

## 4. El hallazgo central: los pips no bastan

La intuición de cualquier jugador es colocarse donde haya más pips. Este gráfico muestra por
qué esa regla falla.

In [ ]:
datos["tiene_par_camino"] = np.where(
    datos["par_camino"] >= 3, "Con madera Y ladrillo", "Sin esa combinación"
)

fig = px.scatter(
    datos.sample(min(3000, len(datos)), random_state=0),
    x="pips_totales", y=OBJETIVO, color="tiene_par_camino",
    opacity=0.45,
    color_discrete_map={"Con madera Y ladrillo": "#3B6D11", "Sin esa combinación": "#C25B3A"},
    title="Pips de la pareja frente a puntos simulados",
    labels={"pips_totales": "Pips de la pareja", OBJETIVO: "Puntos", "tiene_par_camino": ""},
)
fig.update_layout(height=500, legend=dict(orientation="h", y=1.08))
fig.show()

In [ ]:
# A igualdad de pips, ¿cuánto vale tener la combinación?
banda = datos[datos["pips_totales"].between(17, 21)]
comparacion = banda.groupby("tiene_par_camino")[OBJETIVO].agg(["count", "mean", "std"]).round(2)
comparacion.columns = ["parejas", "puntos promedio", "desviación"]
print("Parejas de 17 a 21 pips:\n")
print(comparacion.to_string())

diferencia = (
    comparacion.loc["Con madera Y ladrillo", "puntos promedio"]
    / comparacion.loc["Sin esa combinación", "puntos promedio"] - 1
)
print(f"\nDiferencia: {diferencia:+.0%} a igualdad de producción total")

Mirando el eje horizontal, las nubes apenas suben: saber que una pareja tiene 21 pips en vez de
17 dice poco sobre los puntos que va a sacar.

Mirando verde contra naranja **a la misma altura de pips**, la separación es clara. La
combinación de recursos importa más que la cantidad.

**Este es el argumento del proyecto.** Si los pips predijeran bien, la app sobraría: bastaría
con contar puntitos, que es lo que ya hace todo el mundo. Que no predigan es la razón de que
un modelo aporte algo.

## 5. Colinealidad: variables que dicen lo mismo

Antes de entrenar hay que detectar variables redundantes. Dos variables que miden casi lo mismo
no estropean la predicción, pero sí reparten el efecto entre sus coeficientes y los vuelven
imposibles de interpretar. Y en este proyecto los coeficientes **son** la explicación que ve el
usuario.

In [ ]:
matriz = datos[variables].corr(numeric_only=True).abs()
triangulo = matriz.where(np.triu(np.ones(matriz.shape), k=1).astype(bool))
redundantes = triangulo.stack().sort_values(ascending=False)
redundantes = redundantes[redundantes > 0.85]

print("Pares de variables con correlación mayor a 0.85:\n")
print(redundantes.round(3).to_string())

In [ ]:
fig = px.imshow(
    datos[variables].corr(numeric_only=True),
    color_continuous_scale="RdBu", zmin=-1, zmax=1, aspect="auto",
    title="Matriz de correlación entre variables",
)
fig.update_layout(height=900, width=950)
fig.show()

### Decisión: qué variables se quitan

`control_<recurso>` es `pips_<recurso>` dividido entre el total de ese recurso en el tablero. Como
ese total varía poco de un tablero a otro, las dos variables acaban midiendo casi lo mismo. Se
conservan los `pips_<recurso>`, que son más fáciles de explicar a un jugador.

`aristas_cercanas` y `num_hexagonos` también van casi de la mano: ambas distinguen costa de
centro. Se conserva `num_hexagonos`.

El criterio general: **ante dos variables redundantes, se queda la que sea más fácil de explicar
en voz alta**, porque el usuario final tiene que entender la recomendación.

In [ ]:
descartadas = [c for c in variables if c.startswith("control_") and c != "control_suministro"]
descartadas += ["aristas_cercanas", "hexagonos_distintos"]

seleccionadas = [c for c in variables if c not in descartadas]

print(f"Se descartan {len(descartadas)}: {', '.join(descartadas)}")
print(f"\nQuedan {len(seleccionadas)} variables para el modelo.")

restante = datos[seleccionadas].corr(numeric_only=True).abs()
restante = restante.where(np.triu(np.ones(restante.shape), k=1).astype(bool)).stack()
print(f"Correlación máxima que queda entre variables: {restante.max():.2f}")

## 6. La partición tiene que ser por tablero

Las ~193 parejas de un mismo tablero comparten los mismos 19 hexágonos con los mismos números.
Si unas caen en entrenamiento y otras en prueba, el modelo ve en el entrenamiento información
del tablero que va a evaluar, y la métrica sale inflada.

La partición correcta es **por `tablero_id`**: un tablero entero va a entrenamiento o a prueba,
nunca partido. Aquí se comparan las dos formas de partir.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import r2_score

X, y = datos[seleccionadas], datos[OBJETIVO]

# Partición ingenua, por fila: MAL
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
r2_mal = r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte))

# Partición correcta, por tablero
particion = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
i_tr, i_te = next(particion.split(X, y, groups=datos["tablero_id"]))
r2_bien = r2_score(
    y.iloc[i_te],
    LinearRegression().fit(X.iloc[i_tr], y.iloc[i_tr]).predict(X.iloc[i_te]),
)

print(f"R² con partición por fila (incorrecta): {r2_mal:.3f}")
print(f"R² con partición por tablero (correcta): {r2_bien:.3f}")
print(f"\nDiferencia: {r2_mal - r2_bien:+.3f}")

La diferencia entre las dos particiones resulta **pequeña** con este dataset, y conviene decirlo
en vez de exagerar el riesgo. La razón es que las variables describen la pareja en sí, no el
tablero: un modelo que aprendió que la madera con ladrillo rinde bien aplica igual en un tablero
que no ha visto.

Aun así, la partición por tablero se conserva. Es la que corresponde a cómo se usa el modelo de
verdad: la app recibe tableros nuevos, no parejas sueltas de tableros ya conocidos. Que la
diferencia salga chica es un resultado tranquilizador, no un permiso para partir mal.

## 7. Conclusiones del análisis exploratorio

**El dataset está sano.** Sin valores faltantes, sin infinitos, con la variable objetivo en un
rango coherente con una partida real de Catan.

**Los pips totales predicen mal.** La medida que usa todo jugador en la mesa queda por debajo de
`produccion_efectiva`, que ajusta la producción por lo que de verdad se puede gastar.

**La combinación de recursos vale más que la cantidad.** A igualdad de pips, entre 17 y 21, las
parejas que completan madera con ladrillo sacan alrededor de un 22% más de puntos. Este es el
hallazgo que justifica el producto.

**El desequilibrio hace daño.** Concentrar la producción correlaciona en negativo con los puntos,
aunque el total sea alto.

**Los puertos necesitan la regresión para juzgarse.** Su correlación simple es negativa, pero
está confundida con el hecho de estar en la costa y producir menos. Habrá que mirar el
coeficiente, no la correlación.

**Quedan variables suficientes y poco redundantes** tras retirar las que medían lo mismo, así que
los coeficientes de la regresión serán interpretables. Es la condición para que la app pueda
explicarle al usuario por qué recomienda lo que recomienda.

Siguiente paso: `02_modelo.ipynb`, con la regresión, su evaluación y el agrupamiento de
estrategias.